In [0]:
%pip install yfinance

In [0]:
%restart_python

In [0]:
import yfinance as yf
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

# 1. Definir os ativos da nossa carteira de simulação
tickers = ["ITUB4.SA", "WEGE3.SA", "HMC", "EURUSD=X"]

# 2. Descarregar os dados (Iterar para manter um formato tabular limpo)
dados_lista = []
for ticker in tickers:
    print(f"A descarregar dados para: {ticker}")
    # Descarrega os últimos 2 anos
    df_temp = yf.download(ticker, period="2y").reset_index()
    
    # Achatar as colunas (o yfinance traz colunas multinível em algumas versões)
    if isinstance(df_temp.columns, pd.MultiIndex):
        df_temp.columns = [col[0] for col in df_temp.columns]
        
    df_temp['Ticker'] = ticker
    dados_lista.append(df_temp)

# Juntar tudo num único DataFrame do Pandas
df_consolidado = pd.concat(dados_lista, ignore_index=True)

# Limpar os nomes das colunas (O Spark não gosta de espaços nos nomes das colunas)
df_consolidado.columns = [str(col).replace(" ", "_") for col in df_consolidado.columns]

# 3. Converter para PySpark DataFrame (Aqui entramos no mundo do Big Data)
spark_df = spark.createDataFrame(df_consolidado)

# 4. Adicionar metadados da Camada Bronze (Linhagem básica de dados)
# Isto mostra aos engenheiros seniores que te preocupas com a rastreabilidade
spark_df = spark_df.withColumn("data_ingestao_bronze", current_timestamp())

# Mostrar uma pré-visualização no ecrã
display(spark_df)

# 5. Guardar os dados em formato DELTA LAKE como tabela gerida
table_name = "dados_financeiros_bronze"

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print(f"Sucesso! Dados guardados na camada Bronze como tabela: {table_name}")

In [0]:
from pyspark.sql.functions import col, to_date, round, current_timestamp

# 1. Ler os dados da tabela que você criou na camada Bronze
df_bronze = spark.table("dados_financeiros_bronze")

# 2. Aplicar as Transformações (Limpeza e Qualidade)
df_silver = (df_bronze
    # Padronizar nomes das colunas para português e letras minúsculas (boa prática de governança)
    .withColumnRenamed("Date", "data")
    .withColumnRenamed("Open", "abertura")
    .withColumnRenamed("High", "maxima")
    .withColumnRenamed("Low", "minima")
    .withColumnRenamed("Close", "fechamento")
    .withColumnRenamed("Volume", "volume")
    .withColumnRenamed("Ticker", "ticker")
    
    # Converter a coluna de data para o tipo Date oficial do Spark
    .withColumn("data", to_date(col("data")))
    
    # Arredondar os valores financeiros para 2 casas decimais (ninguém precisa de 6 casas decimais em ações)
    .withColumn("abertura", round(col("abertura"), 2))
    .withColumn("maxima", round(col("maxima"), 2))
    .withColumn("minima", round(col("minima"), 2))
    .withColumn("fechamento", round(col("fechamento"), 2))
    
    # Qualidade de Dados: Remover registros de feriados/fins de semana onde não houve negociação
    .filter(col("volume") > 0)
    .filter(col("fechamento").isNotNull())
    
    # Qualidade de Dados: Remover duplicatas caso o pipeline rode duas vezes no mesmo dia por acidente
    .dropDuplicates(["data", "ticker"])
    
    # Adicionar carimbo de tempo mostrando quando o dado foi limpo
    .withColumn("data_processamento_silver", current_timestamp())
    
    # Selecionar e reordenar apenas as colunas que importam para o negócio
    .select("data", "ticker", "abertura", "maxima", "minima", "fechamento", "volume", "data_processamento_silver")
)

# Mostrar o resultado na tela para validação visual
display(df_silver)

# 3. Salvar como Tabela Gerenciada (Silver) no formato Delta
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dados_financeiros_silver")

print("Sucesso! Dados limpos, padronizados e salvos na camada Silver.")

In [0]:
%sql
-- 1. Criando a tabela Gold utilizando CTEs (Common Table Expressions - o bloco 'WITH')
CREATE OR REPLACE TABLE dados_financeiros_gold AS

WITH cotacoes_diarias AS (
    -- Selecionamos apenas o essencial da camada Silver
    SELECT 
        data,
        ticker,
        fechamento
    FROM dados_financeiros_silver
),

metricas_avancadas AS (
    -- Aplicando Window Functions (Funções de Janela) exigidas na vaga
    SELECT
        data,
        ticker,
        fechamento,
        
        -- Window Function 1: Calculando a média móvel dos últimos 7 dias para suavizar a curva de preços
        AVG(fechamento) OVER (
            PARTITION BY ticker 
            ORDER BY data 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS media_movel_7d,
        
        -- Window Function 2: Pegando o fechamento do dia útil anterior (LAG) para comparar com o dia atual
        LAG(fechamento, 1) OVER (
            PARTITION BY ticker 
            ORDER BY data
        ) AS fechamento_anterior
        
    FROM cotacoes_diarias
)

-- Montando o resultado final já com as regras de negócio aplicadas
SELECT 
    data,
    ticker,
    fechamento,
    ROUND(media_movel_7d, 2) AS media_movel_7d,
    fechamento_anterior,
    -- Matemática simples para calcular a variação % de um dia para o outro
    ROUND(((fechamento - fechamento_anterior) / fechamento_anterior) * 100, 2) AS variacao_percentual_diaria
FROM metricas_avancadas
WHERE fechamento_anterior IS NOT NULL;

In [0]:
%sql
-- Otimizando a tabela para leituras rápidas nos Dashboards (Diferencial da Vaga)
OPTIMIZE dados_financeiros_gold ZORDER BY (ticker, data);

In [0]:
%sql
SELECT * FROM dados_financeiros_gold
ORDER BY ticker, data DESC
LIMIT 20;